# Lumis 19: final model v3 (MedSigLIP), calibration, one-time test, export

Notebook 16 chose **MedSigLIP + extra cases** and notebook 18 kept it unchanged. This notebook:
1. **Fits the final model v3** on all development data: DermNet training photos, all 482 SCIN train + validation
   cases and the 221 extra cases (C = 0.01, the value chosen in every fold of notebook 16).
2. **Calibrates** it on notebook 16's out-of-fold predictions for the 482 cases (never on test photos): one
   temperature, then the confidence bands with the same rule as v1 and v2 (lowest threshold with at least 20 cases
   reaching 85% for "high" and 75% for everything answered; fallback targets 70% to 50%; scabies never "high").
3. **Tests once** on photos never used before: the **121 SCIN test cases** (one close-up each) and the **DermNet test
   split** (972 photos, already used once for v1, so a reused set for v1 and new for v3; disclosed). v1 is scored on
   the same photos; Claude Haiku 5.5 asked directly is scored on the SCIN test cases too.
4. **Sets the photo check** for v3 (the model-dependent "nothing detected" threshold, 99% pass rule as notebook 10).
5. **Exports** the MedSigLIP image encoder to ONNX plus the small classifier, with parity checks.

The test runs once: its results are saved to `exports/final_test_v3/metrics.json`, and a rerun reuses them.

**Runtime: GPU (T4).** Needs Colab secrets `HF_TOKEN` and `ANTHROPIC_API_KEY` (about $0.05 for 121 Claude calls).
About 30 to 45 minutes.

In [ ]:
SEED = 42
C_FINAL = 0.01
PCA_DIM = 256
SCIN_W = 3.0
MAX_WEIGHT = 5.0
TARGET_HIGH, TARGET_ANSWERED = 0.85, 0.75
FALLBACK_TARGETS = [0.70, 0.65, 0.60, 0.55, 0.50]
PASS_TARGET = 0.99                      # photo check: share of real condition photos that must pass
N_BOOT = 2000
ZERO_SHOT_MODEL = 'claude-haiku-5-5'

In [ ]:
!pip install -q onnx onnxruntime anthropic
from google.colab import drive, userdata
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/Lumis'
HF_TOKEN = userdata.get('HF_TOKEN')
ANTHROPIC_KEY = userdata.get('ANTHROPIC_API_KEY')

In [ ]:
import os, io, json, time, zipfile, hashlib, base64
import numpy as np, pandas as pd

dec16 = json.load(open(f'{BASE}/exports/foundation/decision.json'))
dec18 = json.load(open(f'{BASE}/exports/addons/decision.json'))
assert dec16['winner'] == 'medsiglip + extra' and dec18['final'] == 'base', 'This notebook is written for MedSigLIP + extra with no add-ons.'

for z, marker, dest in [(f'{BASE}/data/dermnet.zip', '/content/dermnet', '/content/dermnet'),
                        (f'{BASE}/data/scin_images.zip', '/content/scin', '/content'),
                        (f'{BASE}/data/scin_extra_images.zip', '/content/scin_extra', '/content')]:
    if not os.path.exists(marker):
        with zipfile.ZipFile(z) as f:
            f.extractall(dest)

runs = pd.read_csv(f'{BASE}/exports/runs_log.csv')
V1_RUN = runs[(runs.backbone == 'efficientnetb0') & (runs.split == 'v2')].iloc[-1]['run']
v1info = runs[runs.run == V1_RUN].iloc[-1]
cal1 = json.load(open(f'{BASE}/exports/runs/{V1_RUN}/calibration.json'))
CLASSES = cal1['classes']
K = len(CLASSES)
IDX = {c: i for i, c in enumerate(CLASSES)}
ADJ = np.array(cal1['logit_adjust'])
SCABIES = IDX['scabies']

derm = pd.read_csv(f'{BASE}/data/lumis_split_v2.csv')
derm['file'] = '/content/dermnet/' + derm.path
d_train, d_val, d_test = [derm[derm.split == s].reset_index(drop=True) for s in ('train', 'val', 'test')]
split = pd.read_csv(f'{BASE}/data/scin_split_v1.csv', dtype={'case_id': str})
split['file'] = '/content/' + split.path
tv = split[split.split.isin(['train', 'val'])].reset_index(drop=True)
scin_test = split[(split.split == 'test') & split.eval_photo].reset_index(drop=True)
extra = pd.read_csv(f'{BASE}/data/scin_extra_v1.csv', dtype={'case_id': str})
extra['file'] = '/content/' + extra.path
print(f'DermNet train {len(d_train)} / val {len(d_val)} / test {len(d_test)} | SCIN train+val {len(tv)} photos | '
      f'SCIN test {len(scin_test)} cases | extra {len(extra)} photos')

## 1. MedSigLIP embeddings (development photos from notebook 16; test photos computed here)

In [ ]:
# MEDSIGLIP
import torch
from PIL import Image, ImageOps
from transformers import AutoModel, AutoProcessor
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
_ms = {}

def get_medsiglip():
    if 'm' not in _ms:
        _ms['m'] = AutoModel.from_pretrained('google/medsiglip-448', token=HF_TOKEN).to(DEV).eval()
        _ms['p'] = AutoProcessor.from_pretrained('google/medsiglip-448', token=HF_TOKEN)
    return _ms['m'], _ms['p']

def image_features(model, pixel_values):
    f = model.get_image_features(pixel_values=pixel_values)
    return getattr(f, 'pooler_output', f)

In [ ]:
def open_rgb(path):
    return ImageOps.exif_transpose(Image.open(path)).convert('RGB')

def embed(paths, batch=32):
    model, proc = get_medsiglip()
    out = []
    with torch.no_grad():
        for i in range(0, len(paths), batch):
            x = proc(images=[open_rgb(p) for p in paths[i:i + batch]], return_tensors='pt')
            out.append(image_features(model, x['pixel_values'].to(DEV)).float().cpu().numpy())
    return np.concatenate(out).astype(np.float32)

def readable(files):
    ok = []
    for f in files:
        try:
            Image.open(f).verify()
            ok.append(f)
        except Exception:
            print('cannot be read, left out:', f)
    return set(ok)

# development photos: exactly the list notebook 16 embedded
good = readable(list(d_train.file) + list(d_val.file) + list(tv.file) + list(extra.file))
d_train, d_val = d_train[d_train.file.isin(good)].reset_index(drop=True), d_val[d_val.file.isin(good)].reset_index(drop=True)
extra, tv = extra[extra.file.isin(good)].reset_index(drop=True), tv[tv.file.isin(good)].reset_index(drop=True)
ALL = list(dict.fromkeys(list(d_train.file) + list(d_val.file) + list(tv.file) + list(extra.file)))
KEY = hashlib.md5('\n'.join(ALL).encode()).hexdigest()[:10]
d = f'{BASE}/exports/embeddings/medsiglip'
n_chunks = (len(ALL) + 255) // 256
missing = [k for k in range(n_chunks) if not os.path.exists(f'{d}/{KEY}_{k:04d}.npy')]
assert not missing, f'{len(missing)} MedSigLIP chunks missing: run notebook 16 section 3 first.'
E = np.concatenate([np.load(f'{d}/{KEY}_{k:04d}.npy') for k in range(n_chunks)])
row = {f: i for i, f in enumerate(ALL)}

# test photos (never embedded before)
test_files = list(dict.fromkeys(list(scin_test.file) + list(d_test.file)))
bad = set(test_files) - readable(test_files)
scin_test, d_test = scin_test[~scin_test.file.isin(bad)].reset_index(drop=True), d_test[~d_test.file.isin(bad)].reset_index(drop=True)
test_files = [f for f in test_files if f not in bad]
TKEY = hashlib.md5('\n'.join(test_files).encode()).hexdigest()[:10]
tpath = f'{d}/test_{TKEY}.npy'
if os.path.exists(tpath):
    ET = np.load(tpath)
else:
    t0 = time.time()
    ET = embed(test_files)
    np.save(tpath, ET)
    print(f'Embedded {len(test_files)} test photos in {time.time() - t0:.0f} s')
trow = {f: i for i, f in enumerate(test_files)}
print('development', E.shape, '| test', ET.shape)

## 2. Final model v3 (all development data)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

def balanced(labels, cap=None):
    labels = np.asarray(labels)
    w = len(labels) / (K * np.maximum(np.bincount(labels, minlength=K), 1))
    return (np.minimum(w, cap) if cap else w)[labels]

yd = d_train.lumis_class.map(IDX).values
pool = pd.concat([tv.assign(mult=1.0), extra.assign(mult=extra.agreement.astype(float))])
yp = pool.lumis_class.map(IDX).values.astype(int)
X = E[[row[f] for f in list(d_train.file) + list(pool.file)]]
y = np.concatenate([yd, yp])
w = np.concatenate([balanced(yd), SCIN_W * balanced(yp, MAX_WEIGHT) * pool.mult.values])
scaler = StandardScaler().fit(X)
pca = PCA(PCA_DIM, random_state=SEED).fit(scaler.transform(X))
clf = LogisticRegression(C=C_FINAL, max_iter=3000).fit(pca.transform(scaler.transform(X)), y, sample_weight=w)

def v3_logits(F):
    return clf.decision_function(pca.transform(scaler.transform(F))) + ADJ
print(f'v3 trained on {len(y)} photos | DermNet val accuracy:',
      round(float((v3_logits(E[[row[f] for f in d_val.file]]).argmax(1) == d_val.lumis_class.map(IDX).values).mean()), 4))

## 3. Calibration on notebook 16's out-of-fold predictions (482 cases)

In [ ]:
from scipy.optimize import minimize_scalar
from sklearn.model_selection import KFold

oof = np.load(f'{BASE}/exports/foundation/oof_logits.npz')
z_oof, y_oof = oof['medsiglip_plus_extra'], oof['y']

def softmax(z):
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)

def nll(p, y):
    return float(-np.mean(np.log(p[np.arange(len(y)), y] + 1e-12)))

def ece(p, y, bins=15):
    conf, ok = p.max(1), p.argmax(1) == y
    e = 0.0
    for lo, hi in zip(np.linspace(0, 1, bins + 1)[:-1], np.linspace(0, 1, bins + 1)[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            e += m.mean() * abs(ok[m].mean() - conf[m].mean())
    return float(e)

def fit_T(z, y):
    return float(np.exp(minimize_scalar(lambda lt: nll(softmax(z / np.exp(lt)), y), bounds=(-3, 3), method='bounded').x))

cal_oof = np.zeros_like(z_oof)
for tr, te in KFold(5, shuffle=True, random_state=SEED).split(z_oof):
    cal_oof[te] = softmax(z_oof[te] / fit_T(z_oof[tr], y_oof[tr]))
T = fit_T(z_oof, y_oof)
print(f'Temperature {T:.4f} | ECE raw {ece(softmax(z_oof), y_oof):.3f} -> calibrated (out-of-fold) {ece(cal_oof, y_oof):.3f}')

conf, correct = cal_oof.max(1), cal_oof.argmax(1) == y_oof
def lowest_threshold(target, min_cases=20):
    for t in np.round(np.arange(0.2, 0.99, 0.01), 2):
        m = conf >= t
        if m.sum() >= min_cases and correct[m].mean() >= target:
            return float(t)
    return None
t_high, t_low, used, notes = lowest_threshold(TARGET_HIGH), lowest_threshold(TARGET_ANSWERED), TARGET_ANSWERED, []
if t_high is None:
    notes.append(f'No threshold reaches {TARGET_HIGH:.0%}: no high band.')
if t_low is None:
    for target in FALLBACK_TARGETS:
        t_low = lowest_threshold(target)
        if t_low is not None:
            used = target
            notes.append(f'{TARGET_ANSWERED:.0%} not reachable; fallback {target:.0%} used.')
            break
if t_low is None:
    used = None
    notes.append('Not even 50% reachable: everything referred.')
refer_below = t_low if t_low is not None else 1.01
high_from = max(t_high if t_high is not None else 1.01, refer_below)
BANDS = {'high_from': high_from, 'moderate_from': refer_below, 'refer_below': refer_below, 'answered_target_used': used, 'notes': notes}
print(json.dumps(BANDS, indent=1))

In [ ]:
def bands_of(p, b, cap=True):
    c, pred = p.max(1), p.argmax(1)
    band = np.where(c >= b['high_from'], 'high', np.where(c >= b['refer_below'], 'moderate', 'refer'))
    if cap:
        band = np.where((pred == SCABIES) & (band == 'high'), 'moderate', band)
    return band

def macro_sens(y, p):
    return float(np.mean([(p[y == k] == k).mean() for k in range(K) if (y == k).any()]))

def report(y, p, b):
    pred, band = p.argmax(1), bands_of(p, b)
    ok = pred == y
    from sklearn.metrics import f1_score
    r = {'n': int(len(y)), 'accuracy': float(ok.mean()), 'macro_sensitivity': macro_sens(y, pred),
         'macro_f1': float(f1_score(y, pred, average='macro', labels=range(K), zero_division=0))}
    for k, c in enumerate(CLASSES):
        r[f'sens_{c}'] = float((pred[y == k] == k).mean()) if (y == k).any() else None
        r[f'n_{c}'] = int((y == k).sum())
    shown = band != 'refer'
    r['share_shown'] = float(shown.mean())
    r['accuracy_when_shown'] = float(ok[shown].mean()) if shown.any() else None
    for bn in ('high', 'moderate', 'refer'):
        m = band == bn
        r[f'share_{bn}'] = float(m.mean())
        r[f'accuracy_{bn}'] = float(ok[m].mean()) if m.any() else None
    return r

dev = report(y_oof, cal_oof, BANDS)
print('Development (482 cases, out-of-fold):', {k: round(v, 3) for k, v in dev.items() if isinstance(v, float)})

## 4. One-time test (SCIN test cases and DermNet test split)

In [ ]:
import keras, tensorflow as tf
for gpu in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(gpu, True)

TEST_DIR = f'{BASE}/exports/final_test_v3'
os.makedirs(TEST_DIR, exist_ok=True)
already = os.path.exists(f'{TEST_DIR}/metrics.json')
if already:
    metrics = json.load(open(f'{TEST_DIR}/metrics.json'))
    print('The test was already run once; showing the saved result (nothing is recomputed).')
else:
    v1 = keras.models.load_model(v1info.model_file)
    def v1_probs(files):
        def load(path):
            img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
            img = tf.cast(tf.cast(tf.image.resize(img, (224, 224)), tf.uint8), tf.float32)
            return keras.applications.efficientnet.preprocess_input(img)
        z = v1.predict(tf.data.Dataset.from_tensor_slices(list(files)).map(load).batch(32), verbose=0) + ADJ
        return softmax(z / cal1['temperature'])
    V1_BANDS = cal1['bands']

    metrics, preds = {}, {}
    rng = np.random.default_rng(SEED)
    for name, df in [('scin_test', scin_test), ('dermnet_test', d_test)]:
        yt = df.lumis_class.map(IDX).values
        p3 = softmax(v3_logits(ET[[trow[f] for f in df.file]]) / T)
        p1 = v1_probs(df.file)
        boots = [rng.integers(0, len(yt), len(yt)) for _ in range(N_BOOT)]
        diff = [macro_sens(yt[b], p3[b].argmax(1)) - macro_sens(yt[b], p1[b].argmax(1)) for b in boots]
        acc_d = [(p3[b].argmax(1) == yt[b]).mean() - (p1[b].argmax(1) == yt[b]).mean() for b in boots]
        metrics[name] = {'v3': report(yt, p3, BANDS), 'v1': report(yt, p1, V1_BANDS),
                         'v3_minus_v1_macro_sensitivity_ci95': [float(np.percentile(diff, 2.5)), float(np.percentile(diff, 97.5))],
                         'v3_minus_v1_accuracy_ci95': [float(np.percentile(acc_d, 2.5)), float(np.percentile(acc_d, 97.5))]}
        preds[name] = (yt, p3, p1)
    yt, p3, p1 = preds['scin_test']
    tone = scin_test.tone.fillna('unknown').values
    metrics['scin_test']['by_tone'] = {t: {'n': int((tone == t).sum()), 'v1': float((p1.argmax(1) == yt)[tone == t].mean()),
                                           'v3': float((p3.argmax(1) == yt)[tone == t].mean())} for t in sorted(set(tone))}

In [ ]:
if not already:
    import anthropic
    NAMES = {'acne': 'Acne', 'eczema': 'Eczema (atopic dermatitis)', 'psoriasis': 'Psoriasis', 'scabies': 'Scabies',
             'tinea': 'Fungal skin infection (tinea or ringworm)'}
    BY_NAME = {v: k for k, v in NAMES.items()}
    SCHEMA = {'type': 'object', 'properties': {'condition': {'type': 'string', 'enum': list(NAMES.values())}},
              'required': ['condition'], 'additionalProperties': False}
    PROMPT = ('Someone took this photo of a skin problem with their phone. Which one of these five conditions is it most '
              'consistent with? ' + '; '.join(NAMES.values()) + '. Choose the single most likely one.')      # same as notebook 16
    client = anthropic.Anthropic(api_key=ANTHROPIC_KEY, timeout=60, max_retries=2)

    def ask(path):
        im = open_rgb(path)
        im.thumbnail((1024, 1024), Image.LANCZOS)
        buf = io.BytesIO()
        im.save(buf, 'JPEG', quality=90)
        try:
            r = client.messages.create(model=ZERO_SHOT_MODEL, max_tokens=4096, system='Answer only with the requested JSON.',
                messages=[{'role': 'user', 'content': [
                    {'type': 'image', 'source': {'type': 'base64', 'media_type': 'image/jpeg',
                                                 'data': base64.standard_b64encode(buf.getvalue()).decode()}},
                    {'type': 'text', 'text': PROMPT}]}],
                extra_body={'output_config': {'format': {'type': 'json_schema', 'schema': SCHEMA}}})
            text = next((b.text for b in r.content if getattr(b, 'type', None) == 'text'), None)
            return IDX[BY_NAME[json.loads(text)['condition']]]
        except Exception as e:
            print('failed (counted wrong):', os.path.basename(path), type(e).__name__)
            return -1

    yt, p3, _ = preds['scin_test']
    pc = np.array([ask(f) for f in scin_test.file])
    metrics['scin_test']['claude_asked_directly'] = {'model': ZERO_SHOT_MODEL, 'accuracy': float((pc == yt).mean()),
        'macro_sensitivity': macro_sens(yt, pc), 'failed_calls': int((pc < 0).sum()),
        **{f'sens_{c}': float((pc[yt == k] == k).mean()) if (yt == k).any() else None for k, c in enumerate(CLASSES)}}
    metrics.update({'v3_temperature': T, 'v3_bands': BANDS, 'development_out_of_fold': dev, 'tested_at': time.strftime('%Y-%m-%d %H:%M')})
    json.dump(metrics, open(f'{TEST_DIR}/metrics.json', 'w'), indent=1, default=float)
    print('Saved the one-time test result.')

In [ ]:
rows = []
for name in ('scin_test', 'dermnet_test'):
    for m in ('v1', 'v3'):
        r = metrics[name][m]
        rows.append({'set': name, 'model': m, **{k: r[k] for k in ['n', 'accuracy', 'macro_sensitivity', 'macro_f1',
                     'sens_acne', 'sens_eczema', 'sens_psoriasis', 'sens_scabies', 'sens_tinea', 'share_shown',
                     'accuracy_when_shown', 'share_high', 'accuracy_high']}})
    if 'claude_asked_directly' in metrics[name]:
        r = metrics[name]['claude_asked_directly']
        rows.append({'set': name, 'model': 'Claude asked directly', 'accuracy': r['accuracy'], 'macro_sensitivity': r['macro_sensitivity'],
                     **{f'sens_{c}': r[f'sens_{c}'] for c in CLASSES}})
display(pd.DataFrame(rows).round(3))
for name in ('scin_test', 'dermnet_test'):
    print(f'{name}: v3 minus v1, average sensitivity 95% CI {np.round(metrics[name]["v3_minus_v1_macro_sensitivity_ci95"], 3)}, '
          f'accuracy 95% CI {np.round(metrics[name]["v3_minus_v1_accuracy_ci95"], 3)}')
print('SCIN test cases per condition:', {c: metrics['scin_test']['v3'][f'n_{c}'] for c in CLASSES})
display(pd.DataFrame(metrics['scin_test']['by_tone']).T.round(3))

## 5. Photo check threshold for v3 ("nothing detected", 99% pass rule)

In [ ]:
pc_v1 = json.load(open(f'{BASE}/exports/model/photo_check.json'))
def energy(z):
    m = z.max(1, keepdims=True)
    return -(m[:, 0] + np.log(np.exp(z - m).sum(1)))
real = np.concatenate([energy(v3_logits(E[[row[f] for f in d_val.file]]) / T), energy(z_oof / T)])
max_energy = float(np.quantile(real, PASS_TARGET))
removed = pd.read_csv(f'{BASE}/exports/removed_nonskin_v2.csv')
ns_files = [f for f in ('/content/dermnet/' + removed.path) if os.path.exists(f)]
ns_files = sorted(readable(ns_files))
e_ns = energy(v3_logits(embed(ns_files)) / T) if ns_files else np.array([])
photo_check_v3 = {**pc_v1, 'max_energy': max_energy, 'model': 'v3', 'pass_target': PASS_TARGET,
                  'energy_on': 'v3 logits / temperature; DermNet val + SCIN out-of-fold'}
print(f'Energy threshold {max_energy:.3f} | real photos passing {(real <= max_energy).mean():.1%} | '
      f'DermNet non-skin caught by energy {(e_ns > max_energy).mean():.1%} of {len(e_ns)}')

## 6. Export: ONNX image encoder + classifier, with parity checks

In [ ]:
# EXPORT
import onnxruntime as ort
model, proc = get_medsiglip()
ip = proc.image_processor
PRE = {'size': [int(ip.size['height']), int(ip.size['width'])], 'resample': int(ip.resample),
       'rescale_factor': float(ip.rescale_factor), 'image_mean': [float(v) for v in ip.image_mean],
       'image_std': [float(v) for v in ip.image_std]}
print('Preprocessing:', PRE)

def preprocess_np(img):
    """What the backend does: EXIF-upright RGB -> resize -> scale -> normalize -> NCHW float32."""
    h, w = PRE['size']
    a = np.asarray(img.resize((w, h), resample=PRE['resample']), dtype=np.float32) * PRE['rescale_factor']
    a = (a - np.array(PRE['image_mean'], np.float32)) / np.array(PRE['image_std'], np.float32)
    return a.transpose(2, 0, 1)[None]

sample = list(scin_test.file[:8]) + list(d_test.file[:8])
pre_diff = max(float(np.abs(preprocess_np(open_rgb(f)) - proc(images=[open_rgb(f)], return_tensors='np')['pixel_values']).max())
               for f in sample)
print(f'Preprocessing parity (numpy vs processor): max difference {pre_diff:.2e}')
assert pre_diff < 1e-3, "Preprocessing does not match the processor: send this error to Claude."

class Encoder(torch.nn.Module):
    def __init__(self, m):
        super().__init__()
        self.m = m
    def forward(self, pixel_values):
        return image_features(self.m, pixel_values)

OUT = f'{BASE}/exports/model_v3'
os.makedirs(OUT, exist_ok=True)
onnx_local = '/content/lumis_medsiglip_v3.onnx'
enc = Encoder(model.float().cpu()).eval()
dummy = torch.from_numpy(preprocess_np(open_rgb(sample[0])))
t0 = time.time()
try:
    torch.onnx.export(enc, (dummy,), onnx_local, input_names=['pixel_values'], output_names=['embedding'],
                      dynamic_axes={'pixel_values': {0: 'batch'}, 'embedding': {0: 'batch'}}, opset_version=17, dynamo=False)
except TypeError:
    torch.onnx.export(enc, (dummy,), onnx_local, input_names=['pixel_values'], output_names=['embedding'],
                      dynamic_axes={'pixel_values': {0: 'batch'}, 'embedding': {0: 'batch'}}, opset_version=17)
print(f'ONNX export {time.time() - t0:.0f} s, {os.path.getsize(onnx_local) / 1e9:.2f} GB')

sess = ort.InferenceSession(onnx_local, providers=['CPUExecutionProvider'])
emb_diff, same_pred, t_cpu = 0.0, 0, []
with torch.no_grad():
    for f in sample:
        x = preprocess_np(open_rgb(f))
        t1 = time.time()
        e_onnx = sess.run(None, {'pixel_values': x})[0]
        t_cpu.append(time.time() - t1)
        e_torch = enc(torch.from_numpy(x)).numpy()
        emb_diff = max(emb_diff, float(np.abs(e_onnx - e_torch).max()))
        same_pred += int(v3_logits(e_onnx).argmax() == v3_logits(e_torch).argmax())
print(f'ONNX vs PyTorch: max embedding difference {emb_diff:.2e}, same prediction {same_pred}/{len(sample)}, '
      f'CPU time per photo {np.median(t_cpu):.2f} s')
assert same_pred == len(sample)

In [ ]:
import shutil
head_file = f'{OUT}/lumis_head_v3.npz'
np.savez(head_file, scaler_mean=scaler.mean_.astype(np.float32), scaler_scale=scaler.scale_.astype(np.float32),
         pca_mean=pca.mean_.astype(np.float32), pca_components=pca.components_.astype(np.float32),
         coef=clf.coef_.astype(np.float32), intercept=clf.intercept_.astype(np.float32))

# the classifier rebuilt from the saved arrays must match sklearn exactly
h = np.load(head_file)
def head_np(F):
    Z = ((F - h['scaler_mean']) / h['scaler_scale'] - h['pca_mean']) @ h['pca_components'].T
    return Z @ h['coef'].T + h['intercept'] + ADJ
Fchk = ET[:64]
assert (head_np(Fchk).argmax(1) == v3_logits(Fchk).argmax(1)).all()
print('Classifier parity (numpy vs sklearn): max logit difference', float(np.abs(head_np(Fchk) - v3_logits(Fchk)).max()))

sha = hashlib.sha256(open(onnx_local, 'rb').read()).hexdigest()
shutil.copy(onnx_local, f'{OUT}/lumis_medsiglip_v3.onnx')
config = {'version': 'v3', 'model_version': 'lumis-medsiglip-v3', 'encoder': 'google/medsiglip-448 image encoder (ONNX)',
          'onnx_file': 'lumis_medsiglip_v3.onnx', 'onnx_sha256': sha, 'head_file': 'lumis_head_v3.npz',
          'classes': CLASSES, 'logit_adjust': ADJ.tolist(), 'temperature': T, 'bands': BANDS, 'scabies_never_high': True,
          'preprocessing': {**PRE, 'exif_transpose': True, 'color': 'RGB'}, 'embedding_dim': int(ET.shape[1]),
          'trained_on': {'dermnet_train': int(len(d_train)), 'scin_photos': int(len(tv)), 'scin_extra_photos': int(len(extra))},
          'test': {k: metrics[k]['v3'] for k in ('scin_test', 'dermnet_test')}, 'licence_note':
          'MedSigLIP: Health AI Developer Foundations terms. Training data includes SCIN (SCIN Data Use License).',
          'created_at': time.strftime('%Y-%m-%d %H:%M')}
json.dump(config, open(f'{OUT}/lumis_model_config_v3.json', 'w'), indent=1, default=float)
json.dump(photo_check_v3, open(f'{OUT}/photo_check_v3.json', 'w'), indent=1)
print('Saved to', OUT, os.listdir(OUT))